# 01_中间件

**来源:**
- [LangChain Docs — Middleware Overview](https://docs.langchain.com/oss/python/deepagents/middleware)
- [LangChain Docs — Built-in Middleware](https://docs.langchain.com/oss/python/deepagents/middleware#built-in-middleware)
- [LangChain Docs — Custom Middleware](https://docs.langchain.com/oss/python/deepagents/middleware#custom-middleware)
- [LangChain Docs — Dynamic Prompt](https://docs.langchain.com/oss/python/deepagents/middleware#dynamic-prompt)
- [LangChain Docs — Wrap Model Call](https://docs.langchain.com/oss/python/deepagents/middleware#wrap-model-call)

本 Notebook 全面讲解中间件的概念、`@dynamic_prompt`、`@wrap_model_call`、内置中间件列表以及如何编写自定义中间件。

## 目录

1. [中间件概述](#1-中间件概述)
2. [内置中间件列表](#2-内置中间件列表)
3. [@dynamic_prompt 装饰器](#3-dynamic_prompt-装饰器)
4. [@wrap_model_call 装饰器](#4-wrap_model_call-装饰器)
5. [自定义中间件编写](#5-自定义中间件编写)
6. [中间件应用场景](#6-中间件应用场景)

**参考链接**
- [Middleware 文档](https://docs.langchain.com/oss/python/deepagents/middleware)
- [LangGraph 中间件概念](https://langchain-ai.github.io/langgraph/concepts/high_level/#middleware)
- [Deep Agents Code 中间件](https://docs.langchain.com/docs/deep-agents/code/configuration#middleware)

## 1. 中间件概述

中间件（Middleware）是运行在 Agent 调用链中的钩子，允许你在不修改核心 Agent 逻辑的情况下插入自定义行为。

```text
用户输入
    │
    ▼
┌─────────────────────────────────────┐
│       中间件流水线 (Middleware Pipeline) │
│                                     │
│  1. 前置处理 (Before)               │
│     - 修改系统提示 (dynamic_prompt)  │
│     - 注入上下文                     │
│     - 速率限制                       │
│         │                          │
│         ▼                          │
│  2. 模型调用 (Model Call)           │
│     - 日志记录 (wrap_model_call)    │
│     - 监控                          │
│     - 重试/回退                     │
│         │                          │
│         ▼                          │
│  3. 后置处理 (After)                │
│     - 格式化输出                    │
│     - 内容过滤                      │
│     - 缓存                          │
└─────────────────────────────────────┘
    │
    ▼
最终输出
```

### 中间件的优势

| 优势 | 说明 |
|------|------|
| **可组合性** | 多个中间件可以串联，按顺序执行 |
| **可复用性** | 一次编写，多处使用 |
| **关注点分离** | 横切关注点与核心逻辑解耦 |
| **非侵入式** | 无需修改 Agent 核心代码 |

## 2. 内置中间件列表

Deep Agents 提供了多种内置中间件，涵盖常见的横切关注点：

| 中间件 | 说明 | 装饰器/类 |
|--------|------|----------|
| **Dynamic Prompt** | 运行时动态修改系统提示 | `@dynamic_prompt` |
| **Wrap Model Call** | 包装模型调用，添加额外逻辑 | `@wrap_model_call` |
| **Rate Limiter** | 限制 Agent 调用速率 | `RateLimiterMiddleware` |
| **Timeout** | 设置超时控制 | `TimeoutMiddleware` |
| **Token Counter** | 统计 Token 使用量 | `TokenCounterMiddleware` |
| **Logging** | 日志记录中间件 | `LoggingMiddleware` |
| **Tracing** | 分布式追踪（LangSmith） | `TracingMiddleware` |
| **Auth** | 鉴权中间件 | `AuthMiddleware` |
| **Content Filter** | 内容安全过滤 | `ContentFilterMiddleware` |
| **Cache** | 响应缓存 | `CacheMiddleware` |

### 内置中间件配置表

| 中间件 | 配置项 | 默认值 |
|--------|--------|--------|
| Rate Limiter | `max_calls`, `period_seconds` | 10, 60 |
| Timeout | `seconds` | 30 |
| Token Counter | 无（自动统计） | - |
| Cache | `ttl_seconds`, `max_size` | 300, 100 |
| Content Filter | 自定义过滤规则 | 无 |

## 3. @dynamic_prompt 装饰器

`@dynamic_prompt` 装饰器允许你在运行时动态修改系统提示，根据当前上下文注入相关信息。

In [ ]:
from deepagents import create_deep_agent
from deepagents.middleware import dynamic_prompt
from datetime import datetime

# 使用 @dynamic_prompt 装饰器动态修改提示
@dynamic_prompt
def add_time_context(system_prompt: str) -> str:
    """在系统提示中注入当前时间信息"""
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return f"{system_prompt}\n\n当前时间: {now}"


# 多个动态提示可以串联
@dynamic_prompt
def add_user_context(system_prompt: str) -> str:
    """在系统提示中注入用户上下文"""
    return f"{system_prompt}\n用户语言: 中文"


# 创建带动态提示的 Agent
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    system_prompt="你是一个有用的助手。",
    middleware=[add_time_context, add_user_context],  # 动态提示中间件
)

print("动态提示中间件已创建")
print("每次调用时，系统提示会自动注入当前时间和用户上下文")

### @dynamic_prompt 详解

| 特性 | 说明 |
|------|------|
| **输入** | 当前系统提示字符串 |
| **输出** | 修改后的系统提示字符串 |
| **执行时机** | 每次模型调用前 |
| **顺序** | 按列表顺序依次执行 |
| **数据源** | 可以从数据库、API 或配置中获取数据 |

**使用场景：**
- 注入用户特定信息（姓名、偏好）
- 注入实时数据（时间、天气、股票价格）
- 注入上下文信息（当前页面、项目信息）
- 多语言支持（动态切换语言指令）

## 4. @wrap_model_call 装饰器

`@wrap_model_call` 装饰器用于包装模型调用，在模型调用前后添加额外逻辑（如日志、监控、重试等）。

In [ ]:
from deepagents.middleware import wrap_model_call
import time

# 使用 @wrap_model_call 装饰器
@wrap_model_call
def log_model_call(call_func, messages, **kwargs):
    """记录模型调用的日志"""
    print(f"[中间件] 模型调用开始 | 消息数: {len(messages)}")
    print(f"[中间件] 消息: {messages[-1].content[:50] if messages else 'N/A'}...")

    start = time.time()
    try:
        # 调用实际的模型
        result = call_func(messages, **kwargs)
        elapsed = time.time() - start
        print(f"[中间件] 模型调用完成 | 耗时: {elapsed:.2f}s")
        return result
    except Exception as e:
        elapsed = time.time() - start
        print(f"[中间件] 模型调用失败 | 耗时: {elapsed:.2f}s | 错误: {e}")
        raise


# 带重试机制的包装器
@wrap_model_call
def retry_on_failure(call_func, messages, **kwargs):
    """模型调用失败时自动重试"""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            return call_func(messages, **kwargs)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            print(f"[重试] 第 {attempt + 1} 次失败, 正在重试...")
            time.sleep(1 * (attempt + 1))  # 指数退避


# 创建带中间件的 Agent
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    middleware=[log_model_call],  # 日志中间件
)

print("模型调用包装中间件已创建")

### @wrap_model_call 详解

| 特性 | 说明 |
|------|------|
| **参数1** | `call_func` — 实际调用模型的函数 |
| **参数2** | `messages` — 消息列表 |
| **kwargs** | 其他模型参数（temperature, max_tokens 等） |
| **返回值** | 模型调用的结果 |
| **执行时机** | 每次 LLM 调用时 |

**使用场景：**
- 日志记录和监控
- 错误重试和回退策略
- 性能指标收集
- 模型调用审计
- 动态切换模型（A/B 测试）

## 5. 自定义中间件编写

除了使用装饰器，还可以通过继承 Middleware 基类编写更复杂的中间件。

In [ ]:
from deepagents.middleware import Middleware, MiddlewareContext
from typing import Any, Dict


# 自定义中间件：速率限制
class RateLimiterMiddleware(Middleware):
    """限制 Agent 调用的频率"""

    def __init__(self, max_calls: int = 10, period_seconds: int = 60):
        self.max_calls = max_calls
        self.period_seconds = period_seconds
        self.call_timestamps: list[float] = []

    def before_model_call(self, ctx: MiddlewareContext) -> MiddlewareContext:
        """模型调用前执行"""
        now = __import__("time").time()
        # 清理过期的时间戳
        self.call_timestamps = [
            t for t in self.call_timestamps
            if now - t < self.period_seconds
        ]
        # 检查是否超过限制
        if len(self.call_timestamps) >= self.max_calls:
            raise RuntimeError(
                f"速率限制: {self.period_seconds}秒内最多{self.max_calls}次调用"
            )
        self.call_timestamps.append(now)
        return ctx

    def after_model_call(self, ctx: MiddlewareContext) -> MiddlewareContext:
        """模型调用后执行"""
        return ctx


# 自定义中间件：Token 计数
class TokenCounterMiddleware(Middleware):
    """统计 Token 使用量"""

    def __init__(self):
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0

    def after_model_call(self, ctx: MiddlewareContext) -> MiddlewareContext:
        """模型调用后统计 Token"""
        if ctx.result and hasattr(ctx.result, "usage_metadata"):
            usage = ctx.result.usage_metadata
            self.total_prompt_tokens += usage.get("input_tokens", 0)
            self.total_completion_tokens += usage.get("output_tokens", 0)
            print(
                f"[Token计数] 本轮: 输入={usage.get('input_tokens', 0)}, "
                f"输出={usage.get('output_tokens', 0)}, "
                f"累计: 输入={self.total_prompt_tokens}, "
                f"输出={self.total_completion_tokens}"
            )
        return ctx


print("自定义中间件类已定义")

### 5.1 使用自定义中间件

In [ ]:
# 使用自定义中间件
rate_limiter = RateLimiterMiddleware(max_calls=5, period_seconds=10)
token_counter = TokenCounterMiddleware()

agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    system_prompt="你是一个有用的助手。",
    middleware=[rate_limiter, token_counter],
)

print("自定义中间件已应用到 Agent")

### 5.2 Middleware 基类接口

| 方法 | 说明 | 默认行为 |
|------|------|---------|
| `before_model_call(ctx)` | 模型调用前执行 | 返回 ctx |
| `after_model_call(ctx)` | 模型调用后执行 | 返回 ctx |
| `before_tool_call(ctx)` | 工具调用前执行 | 返回 ctx |
| `after_tool_call(ctx)` | 工具调用后执行 | 返回 ctx |

### MiddlewareContext 字段

| 字段 | 类型 | 说明 |
|------|------|------|
| `ctx.messages` | list | 当前消息列表 |
| `ctx.system_prompt` | str | 系统提示 |
| `ctx.result` | AIMessage \| None | 模型/工具调用结果 |
| `ctx.config` | dict | 运行时配置 |
| `ctx.metadata` | dict | 元数据 |

## 6. 中间件应用场景

### 常见场景对照表

| 场景 | 推荐方式 | 说明 |
|------|---------|------|
| **动态提示注入** | `@dynamic_prompt` | 在系统提示中注入实时信息 |
| **日志记录** | `@wrap_model_call` | 记录每次模型调用的输入输出 |
| **速率限制** | 自定义 Middleware | 控制 API 调用频率 |
| **Token 统计** | 自定义 Middleware | 统计和监控 Token 消耗 |
| **错误重试** | `@wrap_model_call` | 模型调用失败时自动重试 |
| **内容过滤** | 自定义 Middleware | 过滤敏感内容 |
| **A/B 测试** | `@wrap_model_call` | 在不同模型间切换 |
| **缓存** | 内置 CacheMiddleware | 缓存重复查询的结果 |
| **鉴权** | 内置 AuthMiddleware | 验证用户权限 |
| **超时控制** | 内置 TimeoutMiddleware | 防止模型调用挂起 |

### 中间件执行顺序

```text
前置处理 (before_model_call):
  m1.before → m2.before → m3.before → ...

模型调用

后置处理 (after_model_call):
  ... → m3.after → m2.after → m1.after
```

中间件按数组顺序执行前置处理，按逆序执行后置处理（洋葱模型）。

---

**参考链接**
- [Middleware 概述](https://docs.langchain.com/oss/python/deepagents/middleware)
- [内置中间件](https://docs.langchain.com/oss/python/deepagents/middleware#built-in-middleware)
- [自定义中间件](https://docs.langchain.com/oss/python/deepagents/middleware#custom-middleware)
- [LangGraph 中间件概念](https://langchain-ai.github.io/langgraph/concepts/high_level/#middleware)
- [Deep Agents Code 配置](https://docs.langchain.com/docs/deep-agents/code/configuration)